In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, embedding
quality metrics (neighbourhood preservation via trustworthiness / kNN
overlap, plus KMeans silhouette as a "clusterability" probe), and
subsampling helpers. Technique-specific code (PCA/KPCA/t-SNE/UMAP
algorithms and their plots) lives in the per-technique files, NOT here.

"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.manifold import trustworthiness
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from kailash_ml import ExperimentTracker
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4
DEFAULT_QUALITY_NEIGHBOURS = 10
QUALITY_MAX_ROWS = 3000  # trustworthiness is O(n^2) memory — cap the rows

# Behavioural features only. `churned` is an OUTCOME label, not behaviour:
# it is kept on the raw frame for post-hoc profiling but never enters the
# reducer (a standardised 0/1 column would dominate Euclidean structure).
# satisfaction_score (1-5) and num_returns (0-6) are deliberately kept as
# ordinal counts, standardised like the continuous columns.
CUSTOMER_FEATURES = [
    "total_revenue",
    "order_count",
    "avg_order_value",
    "days_since_last_order",
    "customer_tenure_days",
    "satisfaction_score",
    "num_returns",
]

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise the behavioural features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
                     (50,000 x 7 on the shipped dataset)
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling (still holds
                     `churned` and the categorical columns for profiling)
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = list(CUSTOMER_FEATURES)

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING QUALITY — neighbourhood preservation + clusterability
# ════════════════════════════════════════════════════════════════════════
# "Does the reducer preserve structure?" is answered by comparing each
# point's neighbours BEFORE and AFTER the reduction:
#   - trustworthiness (sklearn): penalises points that become neighbours in
#     the embedding although they were far apart in the original space.
#     1.0 = no false neighbours; ~0.5 = random layout.
#   - kNN overlap: average fraction of each point's k original neighbours
#     that are still among its k neighbours in the embedding.
#
# K-means silhouette IN THE EMBEDDING is a different question — "how
# blob-like is the picture?" t-SNE and UMAP deliberately pull points into
# tight blobs, so they inflate silhouette by construction. Treat it as a
# clusterability probe, never as proof that structure was preserved.


def knn_overlap(
    X_high: np.ndarray,
    embedding: np.ndarray,
    n_neighbors: int = DEFAULT_QUALITY_NEIGHBOURS,
) -> float:
    """Mean fraction of original k-NN that survive in the embedding."""
    nn_high = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X_high)
    nn_low = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(embedding)
    idx_high = nn_high.kneighbors(X_high, return_distance=False)[:, 1:]
    idx_low = nn_low.kneighbors(embedding, return_distance=False)[:, 1:]
    shared = [len(set(a) & set(b)) for a, b in zip(idx_high, idx_low)]
    return float(np.mean(shared) / n_neighbors)


def evaluate_embedding(
    X_high: np.ndarray,
    embedding: np.ndarray,
    n_neighbors: int = DEFAULT_QUALITY_NEIGHBOURS,
    max_rows: int = QUALITY_MAX_ROWS,
    random_state: int = RANDOM_STATE,
) -> dict[str, float]:
    """Score an embedding: trustworthiness, kNN overlap, silhouette.

    `X_high` and `embedding` must be row-aligned. At most `max_rows` rows
    are scored (deterministic subsample) because trustworthiness needs the
    full pairwise distance matrix.
    """
    X_high = np.asarray(X_high)
    embedding = np.asarray(embedding)
    if X_high.shape[0] != embedding.shape[0]:
        raise ValueError(
            f"X_high has {X_high.shape[0]} rows but embedding has "
            f"{embedding.shape[0]} — they must be row-aligned"
        )
    if X_high.shape[0] > max_rows:
        rows = subsample_indices(X_high.shape[0], max_rows, random_state)
        X_high, embedding = X_high[rows], embedding[rows]
    return {
        "trustworthiness": float(
            trustworthiness(X_high, embedding, n_neighbors=n_neighbors)
        ),
        "knn_overlap": knn_overlap(X_high, embedding, n_neighbors),
        "silhouette": evaluate_embedding_silhouette(embedding),
    }


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    A CLUSTERABILITY probe ("how separable are K-means blobs in this
    picture?"), not a structure-preservation metric — see the note above.
    Returns -1.0 when only one cluster is found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)


def holdout_indices(
    n_samples: int,
    exclude: np.ndarray,
    n_target: int,
    random_state: int = RANDOM_STATE,
) -> np.ndarray:
    """Deterministic subsample of rows NOT in `exclude`.

    Used for genuine out-of-sample transforms: the rows a reducer is
    applied to must not include the rows it was fitted on.
    """
    remaining = np.setdiff1d(np.arange(n_samples), exclude)
    rng = np.random.default_rng(random_state)
    return rng.choice(remaining, min(n_target, len(remaining)), replace=False)


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every dim-reduction technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_3 lesson logs its sweep + final-fit metrics to a single
# SQLite store so students can compare PCA / Kernel-PCA / t-SNE / UMAP
# embedding-quality runs after the lesson group ends. Mirrors the ex_1
# clustering-zoo pattern; separate DB so dim-reduction has its own
# leaderboard distinct from clustering.

DIMREDUCE_DB = "sqlite:///mlfp04_ex3_dimreduction.db"
EXPERIMENT_NAME = "m4_dimreduction_zoo"


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the dim-reduction ExperimentTracker."""
    tracker = await ExperimentTracker.create(store_url=DIMREDUCE_DB)
    return tracker, EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block. See ``rules/patterns.md`` § "Async Resource Cleanup".
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(name, float(value))
        if series_metrics:
            for name, values in series_metrics.items():
                for step, value in enumerate(values, start=1):
                    await run.log_metric(name, float(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.1: Principal Component Analysis via SVD
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Derive PCA from the Singular Value Decomposition (X = U S V^T)
#   - Read a scree plot and pick n_components by variance threshold
#   - Interpret loadings to assign business meaning to each component
#   - Quantify compression quality via reconstruction error
#   - Recognise when PCA is the right tool (linear, fast, exactly invertible)
#
# PREREQUISITES: MLFP04 Exercise 1 (clustering) + linear algebra basics.
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why PCA = SVD of the centred data matrix
#   2. Build — compute SVD, verify against sklearn.PCA
#   3. Train — scree plot + three component-selection criteria
#   4. Visualise — loadings heatmap + reconstruction error curve
#   5. Apply — customer-analytics compression at a Singapore marketplace
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

from kailash_ml import ModelVisualizer



### Kailash-ML ExperimentTracker — every dim-reduction run logs here


In [ ]:
tracker, exp_name = setup_engines()



## THEORY — PCA is SVD


In [ ]:
# Given a centred (and usually standardised) data matrix X of shape
# (n_samples, n_features), the Singular Value Decomposition gives:
#
#     X = U  S  V^T
#
#   - Columns of V are the principal directions (unit vectors in
#     feature space). These are the "axes of maximum variance".
#   - Singular values s_k encode the strength of each direction.
#     The variance explained by PC_k is  s_k^2 / (n - 1).
#   - Scores (the projection of each sample onto the PCs) = X V = U S.
#
# Everything else about PCA is bookkeeping on top of SVD. sklearn.PCA
# computes exactly this under the hood — we will verify it below.



## TASK 2 — BUILD: compute SVD, derive explained variance


In [ ]:
X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape
print("=== E-commerce customers ===")
print(f"Samples: {n_samples:,}  Features: {n_features}")

U, S, Vt = np.linalg.svd(X, full_matrices=False)

explained_variance = S**2 / (n_samples - 1)
total_variance = explained_variance.sum()
evr = explained_variance / total_variance
cum_evr = np.cumsum(evr)

print(f"\nTotal variance (~n_features={n_features}): {total_variance:.2f}")
print("\nTop 10 principal components:")
print(f"{'PC':>4} {'Sing. val':>12} {'Expl. var %':>14} {'Cumulative %':>14}")
print("-" * 48)
for i in range(min(10, n_features)):
    print(f"{i + 1:>4} {S[i]:>12.4f} {evr[i]:>13.2%} {cum_evr[i]:>13.2%}")

# Cross-check against sklearn's implementation. These MUST agree to ~1e-6;
# if they don't, something is wrong with how X was centred/scaled.
pca_full = PCA(n_components=n_features)
pca_full.fit(X)
max_diff = float(np.abs(pca_full.explained_variance_ratio_ - evr).max())
print(f"\nSVD vs sklearn PCA max diff: {max_diff:.2e}")



### Checkpoint 1


In [ ]:
assert max_diff < 1e-6, f"SVD must match sklearn.PCA, got {max_diff:.2e}"
assert abs(cum_evr[-1] - 1.0) < 1e-6, "cumulative variance must sum to 1"
print("[ok] Checkpoint 1 — SVD-derived PCA matches sklearn\n")



## TASK 3 — TRAIN: scree plot + three selection criteria


In [ ]:
# There is no training loop for PCA — the "training" is the SVD itself.
# The decision we make here is how many components to RETAIN.

n_80 = int(np.searchsorted(cum_evr, 0.80) + 1)
n_90 = int(np.searchsorted(cum_evr, 0.90) + 1)
n_95 = int(np.searchsorted(cum_evr, 0.95) + 1)

# Kaiser: retain components whose eigenvalue > 1 (a component that
# captures more than one original feature's worth of variance).
n_kaiser = int((explained_variance > 1.0).sum())

# Broken-stick: a random partition of total variance would give each PC
# a share of sum_{j=i}^p 1/j / p. Retain PCs that beat that share.
broken_stick = np.array(
    [
        sum(1.0 / j for j in range(i, n_features + 1)) / n_features
        for i in range(1, n_features + 1)
    ]
)
n_broken = int((evr > broken_stick).sum())

print("=== Component-selection criteria ===")
print(f"  80% variance threshold : {n_80} components")
print(f"  90% variance threshold : {n_90} components")
print(f"  95% variance threshold : {n_95} components")
print(f"  Kaiser (eigenvalue>1)  : {n_kaiser} components")
print(f"  Broken-stick           : {n_broken} components")
print(
    f"  Compression at 95%     : "
    f"{n_features} -> {n_95} ({n_features / max(n_95, 1):.1f}x)"
)



### Checkpoint 2


In [ ]:
assert n_80 <= n_90 <= n_95 <= n_features, "monotonic thresholds"
assert 1 <= n_kaiser <= n_features
print("[ok] Checkpoint 2 — variance thresholds + Kaiser + broken-stick\n")



## TASK 4 — VISUALISE: scree + loadings + reconstruction error


In [ ]:
# R9A: visual proof, not just a number. Three plots:
#   (a) scree + cumulative variance
#   (b) loadings for the first 5 PCs
#   (c) reconstruction error vs k

viz = ModelVisualizer()

# (a) Scree plot
fig_scree = viz.training_history(
    {
        "Explained variance %": (evr[:20] * 100).tolist(),
        "Cumulative %": (cum_evr[:20] * 100).tolist(),
    },
    x_label="Principal component",
)
fig_scree.update_layout(title="PCA scree: explained variance by component")
scree_path = OUTPUT_DIR / "01_pca_scree.html"
fig_scree.write_html(str(scree_path))
print(f"Saved: {scree_path}")

# (b) Loadings — which features drive each PC
n_pcs_inspect = min(5, n_features)
loadings = Vt[:n_pcs_inspect, :].T  # (n_features, n_pcs)

print(f"\n=== Loadings on first {n_pcs_inspect} PCs ===")
for i in range(n_pcs_inspect):
    col_norm = float(np.linalg.norm(loadings[:, i]))
    assert abs(col_norm - 1.0) < 1e-6, f"PC{i + 1} must be unit norm"
    abs_l = np.abs(loadings[:, i])
    top = np.argsort(abs_l)[::-1][:3]
    names = [f"{feature_cols[j]} ({loadings[j, i]:+.2f})" for j in top]
    print(f"  PC{i + 1}: {', '.join(names)}")

# Loadings heatmap — rows = original features, columns = PCs. Strong
# red/blue cells are the features that define each component.
fig_load = go.Figure(
    data=go.Heatmap(
        z=loadings,
        x=[f"PC{i + 1}" for i in range(n_pcs_inspect)],
        y=feature_cols,
        colorscale="RdBu",
        zmid=0.0,
        colorbar=dict(title="loading"),
    )
)
fig_load.update_layout(title="PCA loadings: which features define each component")
loadings_path = OUTPUT_DIR / "01_pca_loadings_heatmap.html"
fig_load.write_html(str(loadings_path))
print(f"Saved: {loadings_path}")

# (c) Reconstruction error as a function of k
# For standardised data, MSE(k) = sum_{j>k} s_j^2 / (n_samples * n_features)
n_range = list(range(1, min(n_features + 1, 21)))
recon_errors = []
for k in n_range:
    pca_k = PCA(n_components=k, random_state=42)
    X_proj = pca_k.fit_transform(X)
    X_hat = pca_k.inverse_transform(X_proj)
    recon_errors.append(float(np.mean((X - X_hat) ** 2)))

fig_recon = viz.training_history(
    {"Reconstruction MSE": recon_errors}, x_label="Components retained"
)
fig_recon.update_layout(title="PCA: reconstruction error vs components")
recon_path = OUTPUT_DIR / "01_pca_reconstruction.html"
fig_recon.write_html(str(recon_path))
print(f"Saved: {recon_path}")



### Checkpoint 3


In [ ]:
assert recon_errors[0] > recon_errors[-1], "MSE must fall as k grows"
assert recon_errors[-1] < 0.1, "full-rank reconstruction should be ~0"
print("\n[ok] Checkpoint 3 — visualisations + loadings + reconstruction\n")



## TASK 5 — APPLY: Customer-Analytics Compression at a Singapore Marketplace


In [ ]:
# SCENARIO (illustrative): a large Singapore e-commerce marketplace keeps
# millions of active shoppers, each described by 100+ behavioural features
# (category views, basket size, delivery windows, coupon response,
# returns, support tickets...). The analytics team runs a nightly K-means
# segmentation to power a homepage recommendation carousel — but the
# full-dimensional distance computation misses the pre-dawn batch window.
#
# WHY PCA IS THE RIGHT TOOL HERE:
#   - Linear and fast: one SVD (or a randomised SVD) on the nightly batch.
#   - Exactly invertible (up to the discarded components): any customer
#     can be reconstructed from their compressed vector, which matters for
#     downstream churn-model explainability.
#   - Interpretable: loadings let marketing name the components
#     (see the heatmap — e.g. "PC1 = spend volume").
#
# BUSINESS IMPACT (illustrative assumptions, not reported figures):
# K-means cost grows linearly with the number of dimensions, so cutting
# dimensions by the compression ratio printed below cuts the clustering
# step by roughly the same factor. If that brings the nightly job inside
# its window, the carousel refreshes on time every day; put your own
# value on a fresh vs stale carousel to size the benefit.
#
# The variance PCA discards is the smallest-variance directions. Often
# that is mostly noise, which can make downstream segmentation more
# stable — but check it: a small-variance direction can still carry a
# rare, important signal.

# Demonstrate the compression numerically.
compression_ratio = n_features / max(n_95, 1)
print("=== Marketplace-style compression estimate ===")
print(f"  Raw dimensions       : {n_features}")
print(f"  At 95% variance      : {n_95}")
print(f"  Compression ratio    : {compression_ratio:.1f}x")
print(f"  Variance discarded   : {(1 - cum_evr[n_95 - 1]) * 100:.2f}%")
print(
    f"  Reconstruction MSE   : {recon_errors[min(n_95 - 1, len(recon_errors) - 1)]:.4f}"
)



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
# Every M4 ex_3 lesson logs into the SAME experiment ('m4_dimreduction_zoo')
# so PCA / Kernel-PCA / t-SNE / UMAP can be compared side-by-side from one
# SQLite store after the lesson group ends. Sweep series = per-k explained-
# variance + reconstruction-MSE; scalar metrics = the chosen-k headlines.

track_run(
    tracker,
    exp_name,
    run_name="pca_svd",
    params={
        "algorithm": "pca",
        "method": "svd",
        "n_features": n_features,
        "n_samples": n_samples,
        "n_components_chosen": n_95,
        "variance_threshold": 0.95,
    },
    scalar_metrics={
        "n_components_80": float(n_80),
        "n_components_90": float(n_90),
        "n_components_95": float(n_95),
        "n_components_kaiser": float(n_kaiser),
        "n_components_broken_stick": float(n_broken),
        "compression_ratio_at_95": float(compression_ratio),
        "variance_kept_at_95": float(cum_evr[n_95 - 1]),
        "recon_mse_at_95": float(recon_errors[min(n_95 - 1, len(recon_errors) - 1)]),
        "svd_vs_sklearn_max_diff": float(max_diff),
    },
    series_metrics={
        "explained_variance_ratio": evr.tolist(),
        "cumulative_explained_variance": cum_evr.tolist(),
        "reconstruction_mse": recon_errors,
    },
)
print(f"  [tracked] PCA scree + reconstruction series logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — the kailash-ml DimReductionEngine


In [ ]:
# This lesson hand-rolled the SVD, the explained-variance bookkeeping,
# the three component-selection criteria, and the reconstruction-error
# curve — ~140 lines of structure to internalise PCA from first principles.
#
# kailash-ml ships a single engine that IS that pipeline. DimReductionEngine.
# `reduce` runs the whole fit-and-transform on a polars DataFrame and
# returns a DimReductionResult with the embedding, explained variance, and
# reconstruction error. One sync call. The engine owns pca, tsne, umap,
# nmf — the next four lessons all pivot on this same surface.

import polars as pl

from kailash_ml.engines.dim_reduction import DimReductionEngine

cust_df = pl.from_numpy(X, schema=feature_cols)
dimreduce = DimReductionEngine()

reduce_result = dimreduce.reduce(cust_df, algorithm="pca", n_components=n_95)
print(
    f"  DimReductionEngine.reduce(pca, n_components={n_95}): "
    f"embedding shape=({len(reduce_result.transformed)}, "
    f"{reduce_result.n_components})  "
    f"recon_mse={reduce_result.reconstruction_error:.4f}"
)
if reduce_result.explained_variance_ratio is not None:
    cum = sum(reduce_result.explained_variance_ratio)
    print(f"  Explained variance retained: {cum:.2%}")
print()
print("  Every metric you computed by hand — explained variance ratio,")
print("  cumulative variance, reconstruction MSE — is a field on")
print("  DimReductionResult. The engine IS the destination.\n")



## REFLECTION


[x] Derived PCA directly from X = U S V^T and verified against sklearn
  [x] Built a scree plot and applied three selection criteria
  [x] Read loadings to assign business meaning to principal components
  [x] Measured reconstruction error as a compression quality metric
  [x] Drew a loadings heatmap to name the components
  [x] Framed PCA for a nightly marketplace segmentation (illustrative)

  KEY INSIGHT: PCA's inverse is EXACT linear algebra: X_hat = Z V^T + mean,
  and the only error is the variance you chose to discard. Kernel PCA and
  UMAP offer only APPROXIMATE (learned) inverses, and t-SNE has none. If
  your downstream job needs to EXPLAIN a customer in the original feature
  space, PCA is the safest answer.

  Next: 02_kernel_pca.py lifts this into nonlinear territory via the
  kernel trick — same math, richer feature space.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)


# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

